# 🚁 Notebook 01 — Introduction to Dynamic Systems

### The very first building block of the whole course

Welcome! By the end of this course you will build a **1D drone that hovers by itself** using a
PID controller — and, more importantly, you'll understand *exactly why* every part works.

But we don't start with drones or controllers. We start with the single most important idea
underneath all of it:

> **A dynamic system is anything that changes over time.**

That's it. A falling ball, a moving car, a heating room, a drone climbing — all dynamic systems.
Once you can *simulate* how something changes over time, everything else becomes possible.

---

## 🎯 Learning objectives
After this notebook you will be able to:
- Explain what a **dynamic system**, **state**, and **state variable** are — in plain words.
- Describe the chain **acceleration → velocity → position** intuitively.
- Understand what a **time step** (`dt`) is and why we take tiny steps.
- Implement **Euler integration** yourself, by hand, line by line.
- Simulate motion and **animate** it.
- See *why* the size of `dt` changes how accurate your simulation is.

## 🧩 Prerequisites
- Basic Python (variables, `for` loops, lists). That's genuinely all.
- **No calculus, no physics, no differential equations required.** We build every idea from zero.

## ⏱️ Estimated time
About **45–70 minutes** if you run every cell and try the exercises.

## 🧠 Concepts you know so far
- Nothing yet — this is Notebook 1! We start from the very beginning.

## 🔜 Concepts you'll learn in this notebook
Position · Velocity · Acceleration · Time & time steps · State variables · Euler integration · Simulation · Animation


### 📌 How to use this notebook

1. Run cells **top to bottom** (Shift+Enter runs a cell).
2. Always run the **Setup cell** just below first.
3. Where you see 🧪 **Exercises**, actually try them — hints and solutions are hidden inside
   little `▸ Reveal` triangles so you don't spoil yourself.
4. If an animation looks blank, just **re-run that cell once** (Colab sometimes needs a second try).

Let's go.


In [ ]:
# ============================================================
# SETUP CELL  -  run this first, every time
# ============================================================
# We only use numpy, matplotlib and ipywidgets in this whole course.
import numpy as np                      # numbers and arrays
import matplotlib.pyplot as plt         # plotting / graphs
from matplotlib import animation        # for animations
from IPython.display import HTML        # to show animations in the notebook

# Make animations play as an embedded video/player (works in Google Colab).
plt.rcParams["animation.html"] = "jshtml"
plt.rcParams["animation.embed_limit"] = 80   # MB, so longer animations don't get cut off
plt.rcParams["figure.figsize"] = (8, 4)      # a comfortable default plot size

# ipywidgets gives us interactive sliders.
from ipywidgets import interact, FloatSlider, IntSlider

# This line helps sliders/widgets show up correctly in Google Colab.
# It does nothing (harmlessly) if you are NOT in Colab.
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

print("Setup complete. numpy, matplotlib and ipywidgets are ready to go!")


## 1. What is a *dynamic system*? (pure intuition first)

Forget equations. Think about your everyday life:

- ☕ A hot coffee slowly **cools down** over minutes.
- 🚗 A car **speeds up** when you press the gas.
- 🎈 A balloon you let go **drifts** across the room.
- 🚁 A drone **rises** when its motors spin faster.

Every one of these is a **dynamic system**: its situation *right now* depends on what was
happening *a moment ago*. The coffee is 60°C now because it was 61°C a moment ago and lost a
little heat.

To describe a dynamic system at any instant, we ask: **"What numbers do I need to know to fully
describe its situation right now?"** Those numbers are called the **state**, and each number is a
**state variable**.

For a drone moving straight up and down, the state is just two numbers:

| State variable | Meaning | Symbol |
|---|---|---|
| **Position** | how high it is | `x` |
| **Velocity** | how fast it's moving up/down | `v` |

If you know the drone's height and its speed right now, you know its state. Everything in this
course is about tracking how that state **changes over time**.


## 2. Position — *where* something is

**Position** just answers: *where is it?* For our 1D drone, position is its **altitude** (height
above the ground), measured in **metres (m)**. We'll call it `x`.

- `x = 0` → on the ground
- `x = 5` → 5 metres up
- `x = 10` → 10 metres up

Let's imagine a drone sitting at three different heights over time and simply plot it. No motion
yet — just the idea of position as a number that we can draw.


In [ ]:
# Three snapshots of a drone's height (altitude) at three moments in time.
times   = [0, 1, 2]        # seconds
heights = [2, 5, 8]        # metres  (position x at each time)

print("At each moment, the drone's POSITION (altitude) is just one number:")
for t, x in zip(times, heights):
    print(f"  time = {t} s   ->   position x = {x} m")

# Draw it: time on the horizontal axis, height on the vertical axis.
plt.figure()
plt.plot(times, heights, "o-", color="royalblue", markersize=10)
plt.xlabel("time (seconds)")
plt.ylabel("position x = altitude (metres)")
plt.title("Position is just 'how high' at each moment in time")
plt.grid(True, alpha=0.3)
plt.show()


## 3. Velocity — *how fast* the position is changing

Now the key jump. **Velocity** answers: *how fast is the position changing, and in which
direction?* Units: **metres per second (m/s)**. We'll call it `v`.

Intuition with no math:
- If the drone climbs **2 metres every second**, its velocity is `v = +2 m/s`.
- If it drops **3 metres every second**, its velocity is `v = -3 m/s` (negative = going down).
- If it just sits there, `v = 0`.

So velocity is simply **"change in position, per second."** In fact, that's the whole definition:

$$ v \approx \frac{\text{change in position}}{\text{change in time}} = \frac{\Delta x}{\Delta t} $$

Don't be scared of the symbols — the Greek letter Δ ("delta") just means *"change in."* The
formula says exactly what we said in words: velocity is how much position changed divided by how
much time passed.

Let's flip it around. If we **know the velocity**, we can figure out the future position: to get
the new position, take the old position and add "velocity × time elapsed." Let's simulate a drone
climbing at a steady 2 m/s.


In [ ]:
# A drone climbing at a STEADY velocity of 2 m/s, starting from the ground (x = 0).
velocity = 2.0     # m/s  (constant, doesn't change here)
dt       = 1.0     # seconds between each snapshot ("time step")

x = 0.0            # start on the ground
time_list = []     # we'll record time here
x_list    = []     # ...and position here, so we can plot afterwards

print("Watch position grow because velocity keeps pushing it up:")
for step in range(6):                 # take 6 steps of 1 second each
    t = step * dt
    print(f"  time = {t:.0f}s   position x = {x:.1f} m   (velocity = {velocity} m/s)")
    time_list.append(t)
    x_list.append(x)
    # KEY LINE: new position = old position + velocity * time_step
    x = x + velocity * dt

plt.figure()
plt.plot(time_list, x_list, "o-", color="seagreen", markersize=9)
plt.xlabel("time (s)");  plt.ylabel("position x (m)")
plt.title("Constant velocity (2 m/s) -> position rises in a straight line")
plt.grid(True, alpha=0.3);  plt.show()


👀 **Notice:** the position went up in a perfectly **straight line**. That's the fingerprint of
*constant velocity*: equal steps up, every second. The slope (steepness) of this line *is* the
velocity. A steeper line = faster drone.


## 4. Acceleration — *how fast the velocity is changing*

One more level up, and this is the last one. **Acceleration** answers: *how fast is the velocity
changing?* Units: **metres per second, per second (m/s²)**. We'll call it `a`.

Real-life feel for it:
- Press the gas pedal in a car → you feel pushed back into your seat → that push **is**
  acceleration (your velocity is increasing).
- Gravity pulls everything down at about **9.8 m/s²** — every second you fall, you gain about
  9.8 m/s of downward speed.

So we have a neat **ladder of three ideas**, each one describing the rate of change of the one
below it:

```
   acceleration  a   =  how fast VELOCITY changes   (per second)
        │  adds to
        ▼
     velocity    v   =  how fast POSITION changes   (per second)
        │  adds to
        ▼
     position    x   =  where the thing actually is
```

The magic we'll use for the *entire rest of this course* is this: if we know the **acceleration**,
we can update velocity; and once we have velocity, we can update position. Two tiny additions,
repeated over and over. Let's see it.


In [ ]:
# A drone with constant UPWARD acceleration of 1 m/s^2, starting from rest.
acceleration = 1.0    # m/s^2 (constant)
dt           = 1.0    # seconds per step

x = 0.0               # start position (on ground)
v = 0.0               # start velocity (not moving yet)

t_list, x_list, v_list = [], [], []

print("Acceleration feeds velocity, and velocity feeds position:")
for step in range(7):
    t = step * dt
    print(f"  t={t:.0f}s   v={v:5.1f} m/s   x={x:5.1f} m")
    t_list.append(t); x_list.append(x); v_list.append(v)

    # TWO update lines -- this pattern is the heart of the whole course:
    v = v + acceleration * dt   # 1) acceleration changes the velocity
    x = x + v * dt              # 2) velocity changes the position

# Plot velocity and position side by side.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(t_list, v_list, "o-", color="darkorange"); ax1.set_title("velocity rises in a straight line")
ax1.set_xlabel("time (s)"); ax1.set_ylabel("velocity (m/s)"); ax1.grid(True, alpha=0.3)
ax2.plot(t_list, x_list, "o-", color="crimson");     ax2.set_title("position curves upward!")
ax2.set_xlabel("time (s)"); ax2.set_ylabel("position (m)"); ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()


👀 **Notice the difference from before:** with constant *velocity* the position was a straight
line. With constant *acceleration*, the velocity is a straight line, but the **position curves
upward** — it climbs faster and faster because the velocity keeps growing. That upward curve is
exactly how a falling object behaves (just downward), and we'll use it for gravity in Notebook 02.


## 5. Time steps (`dt`) and **Euler integration** — the engine of every simulation

You've actually already used the trick — let's name it and understand *why* it works.

A computer can't watch something move *continuously*. Instead, it **freezes time into tiny slices**
and updates the numbers slice by slice. The length of one slice is the **time step**, written `dt`
(delta-t). Think of it like the frames of a movie: each frame is a snapshot, and playing them fast
makes smooth motion.

At each little step we do the same two updates you saw above:

```
new velocity = old velocity + acceleration * dt
new position = old position + new velocity  * dt
```

That's it. This "take a tiny step in the direction you're currently heading, then look again"
method has a name: **Euler integration** (named after the mathematician Leonhard Euler, pronounced
"OY-ler"). It is the simplest way to simulate *any* dynamic system, and we'll use it everywhere.

### Why does it work? A walking analogy 🚶
Imagine walking in fog and you can only see one step ahead. You look at the direction you're
facing (that's your *velocity*), take **one small step** that way (`dt`), then stop and look again.
The environment might have nudged you (that's *acceleration* — gravity, thrust, wind…). You adjust
your facing and take another small step. Repeat thousands of times and you trace out a whole path.
**Small steps + constant re-checking = simulation.**

The smaller each step (`dt`), the more carefully you're checking, and the more accurate your path.
We'll *see* this next.


In [ ]:
def simulate_constant_accel(acceleration, dt, total_time):
    """Simulate 1D motion under constant acceleration using Euler integration.
    Returns three lists: times, positions, velocities."""
    x = 0.0                     # start position
    v = 0.0                     # start velocity
    t = 0.0                     # start time

    times, xs, vs = [], [], []
    n_steps = int(total_time / dt)   # how many little steps fit in the total time

    for _ in range(n_steps):
        times.append(t); xs.append(x); vs.append(v)   # record BEFORE updating
        v = v + acceleration * dt    # Euler step 1: update velocity
        x = x + v * dt               # Euler step 2: update position
        t = t + dt                   # advance the clock
    return times, xs, vs

# Run it and print a few rows.
t_list, x_list, v_list = simulate_constant_accel(acceleration=1.0, dt=0.5, total_time=5.0)
print("First few simulated moments (a=1 m/s^2, dt=0.5s):")
for i in range(5):
    print(f"  t={t_list[i]:.1f}s   v={v_list[i]:.2f} m/s   x={x_list[i]:.2f} m")
print(f"... total of {len(t_list)} tiny steps were computed.")


### 🔬 Does the step size matter? Let's *prove* it visually

For constant acceleration there happens to be an exact "true answer" (the smooth curve reality
would follow): position after time *t* is `½ · a · t²`. We don't need to derive it — we'll just
plot it as a black dashed line and compare our Euler simulation against it for a **big** `dt` vs a
**small** `dt`. This shows you, with your own eyes, why small steps are more accurate.


In [ ]:
a = 1.0            # acceleration, m/s^2
total_time = 5.0   # seconds

# The exact, "perfect" answer reality would follow (we just plot it for comparison):
t_true = np.linspace(0, total_time, 200)
x_true = 0.5 * a * t_true**2        # this is the smooth truth for constant acceleration

plt.figure()
plt.plot(t_true, x_true, "k--", linewidth=2, label="true smooth answer")

# Euler with a BIG step vs a SMALL step:
for dt, colour in [(1.0, "red"), (0.1, "green")]:
    tL, xL, _ = simulate_constant_accel(a, dt, total_time)
    plt.plot(tL, xL, "o-", color=colour, markersize=4, label=f"Euler, dt = {dt}s")

plt.xlabel("time (s)"); plt.ylabel("position (m)")
plt.title("Bigger dt = rougher, less accurate.  Smaller dt = closer to reality.")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

print("Big steps (red) cut corners and drift off the true curve.")
print("Small steps (green) hug the true black curve almost perfectly.")


🧠 **Big takeaway:** Euler integration is *approximate*. Smaller `dt` → more accurate but more
computation. Larger `dt` → faster but rougher (and, as you'll see later, can even make a
simulation *explode* into nonsense). Choosing `dt` is a real engineering trade-off. For this
course, `dt = 0.02` (i.e. 50 updates per second) is a nice sweet spot.


## 6. Let's *see* it move — our first animation 🎬

Numbers and static graphs are fine, but you learn best visually — so let's actually watch a dot
(our drone) rise. The animation below simulates constant upward acceleration and draws the drone's
height frame by frame, with a live readout of its position and velocity.

*(If it appears blank, re-run the cell once.)*


In [ ]:
# Simulate the motion we want to animate.
a, dt, total_time = 1.0, 0.05, 6.0
t_list, x_list, v_list = simulate_constant_accel(a, dt, total_time)

# To keep the animation light, only show ~120 evenly spaced frames.
frame_ids = np.linspace(0, len(t_list) - 1, 120).astype(int)

fig, ax = plt.subplots(figsize=(4, 6))
ax.set_xlim(-1, 1)
ax.set_ylim(0, max(x_list) * 1.1 + 1)
ax.set_xticks([])                       # sideways position is meaningless in 1D
ax.set_ylabel("altitude x (m)")
ax.set_title("Drone accelerating upward")
ax.axhline(0, color="saddlebrown", linewidth=3)          # the ground
drone_dot, = ax.plot([], [], "o", color="royalblue", markersize=26)
readout = ax.text(-0.9, ax.get_ylim()[1]*0.92, "", fontsize=11)

def init():
    drone_dot.set_data([], [])
    readout.set_text("")
    return drone_dot, readout

def update(frame):
    i = frame_ids[frame]
    drone_dot.set_data([0], [x_list[i]])                 # move the dot to current height
    readout.set_text(f"t = {t_list[i]:.2f} s\nx = {x_list[i]:.2f} m\nv = {v_list[i]:.2f} m/s")
    return drone_dot, readout

ani = animation.FuncAnimation(fig, update, frames=len(frame_ids),
                              init_func=init, blit=True, interval=40)
plt.close(fig)     # stop a duplicate static image from appearing
HTML(ani.to_jshtml())


▶️ Press play. Watch how the dot starts slow and then races upward — that speeding-up is
**acceleration** feeding **velocity** feeding **position**, exactly the ladder from Section 4,
now alive on screen.


## 7. Play with it yourself — interactive slider 🎛️

Time to experiment. Drag the sliders and the simulation instantly re-runs. Build a *feel* for how
`acceleration` and `dt` change the motion. Try these:
- Set acceleration to a big number → the curve shoots up fast.
- Set acceleration to 0 → flat line (nothing pushing it).
- Make `dt` large (like 0.9) → watch the coloured Euler dots drift away from the true black curve.


In [ ]:
def explore(acceleration=1.0, dt=0.2, total_time=5.0):
    # Our Euler simulation:
    tL, xL, _ = simulate_constant_accel(acceleration, dt, total_time)
    # The exact truth for comparison:
    t_true = np.linspace(0, total_time, 200)
    x_true = 0.5 * acceleration * t_true**2

    plt.figure(figsize=(8, 4.5))
    plt.plot(t_true, x_true, "k--", linewidth=2, label="true answer")
    plt.plot(tL, xL, "o-", color="mediumvioletred", markersize=5, label=f"Euler (dt={dt})")
    plt.xlabel("time (s)"); plt.ylabel("position (m)")
    plt.title(f"acceleration = {acceleration} m/s^2   |   dt = {dt} s")
    plt.legend(); plt.grid(True, alpha=0.3); plt.ylim(bottom=0); plt.show()

interact(explore,
         acceleration=FloatSlider(min=0.0, max=5.0, step=0.5, value=1.0),
         dt=FloatSlider(min=0.05, max=0.9, step=0.05, value=0.2),
         total_time=FloatSlider(min=2.0, max=10.0, step=1.0, value=5.0));


## ✅ Summary — what you now understand

- A **dynamic system** is anything that changes over time; its **state** is the set of numbers
  needed to describe it right now. For 1D motion the state is **position `x`** and **velocity `v`**.
- The **ladder**: acceleration changes velocity, velocity changes position.
- A **time step `dt`** slices time into tiny pieces. **Euler integration** updates the state one
  tiny step at a time using two additions:
  `v += a*dt`  then  `x += v*dt`.
- Constant velocity → straight-line position. Constant acceleration → upward-curving position.
- **Smaller `dt` = more accurate** simulation (but more steps to compute).

You just built the simulation engine that the *entire* drone course runs on. 🎉


## ⚠️ Common mistakes to avoid
- **Forgetting units.** Position is in metres, velocity in m/s, acceleration in m/s². Mixing them
  up quietly breaks everything.
- **Updating position with the *old* velocity vs new velocity.** Different textbooks order these
  two lines differently ("semi-implicit" vs "explicit" Euler). For small `dt` it barely matters;
  we consistently update `v` first, then `x`. Just be consistent.
- **Choosing `dt` too big.** Your simulation gets inaccurate — and later, with controllers, it can
  blow up entirely. When something looks crazy, *shrink `dt` first.*
- **Thinking Euler is exact.** It's an approximation. It's close, not perfect.

## 🧭 Mental model to carry forward
> *"Simulation = take the current state, nudge it by a tiny time step in the direction it's
> currently heading, then look again. Repeat thousands of times."*

## 🌍 Where this shows up in the real world
Video game physics, weather forecasting, spacecraft trajectory planning, animation studios,
epidemic models, and — of course — every flight controller inside every real drone. They all march
a state forward in tiny time steps, exactly like you just did.


## 🧪 Exercises

Try each one *before* revealing the hint or solution. Struggling a little is where the learning
happens.

---
**Level 1 — Observe.** In the animation cell (Section 6), change `a = 1.0` to `a = 3.0` and re-run.
*Before running,* predict: does the drone reach the top faster or slower?

<details><summary>▸ Hint</summary>
More acceleration means velocity grows faster, so position grows faster too.
</details>
<details><summary>▸ Solution</summary>
Faster. Tripling the acceleration makes the drone climb much more quickly — it reaches the top of
the plot in noticeably less time.
</details>

---
**Level 2 — Modify.** Change `simulate_constant_accel` so the drone starts with an initial upward
velocity of **5 m/s** (instead of 0). Re-run the interactive plot. What changes at time 0?

<details><summary>▸ Hint</summary>
Look for the line `v = 0.0` at the start of the function and give it a different starting value.
</details>
<details><summary>▸ Solution</summary>
Set `v = 5.0` at the start. Now even at t=0 the position is already climbing (the curve leaves the
origin with an upward slope instead of being flat), because there's velocity from the very first
instant.
</details>

---
**Level 3 — Predict, then check.** With `a = 2` m/s² and starting from rest, roughly how high is
the drone after 3 seconds? Write down your guess, *then* run
`simulate_constant_accel(2.0, 0.01, 3.0)` and look at the last position.

<details><summary>▸ Hint</summary>
The true answer for constant acceleration from rest is <code>0.5 * a * t*t</code>.
</details>
<details><summary>▸ Solution</summary>
0.5 × 2 × 3² = 0.5 × 2 × 9 = <b>9 metres</b>. Your Euler simulation with a small dt will land very
close to 9 m. If you used a big dt it'd be a bit off — that's the accuracy lesson again.
</details>


## ❓ Quick quiz (concept check)

**Q1.** If velocity is constant and positive, the position-vs-time graph is a…
<details><summary>▸ Answer</summary>
…**straight line** sloping upward. Constant velocity means equal position gains each second.
</details>

**Q2.** Acceleration is the rate of change of ______.
<details><summary>▸ Answer</summary>
**Velocity.** (And velocity is the rate of change of position.)
</details>

**Q3.** You halve `dt`. Your Euler simulation becomes…
<details><summary>▸ Answer</summary>
**More accurate** (closer to the true curve), at the cost of twice as many steps to compute.
</details>

**Q4.** True or false: Euler integration gives the exact, perfect answer.
<details><summary>▸ Answer</summary>
**False.** It's an approximation — very good with small `dt`, but never exactly perfect.
</details>


## 🔭 Preview of Notebook 02 — *Forces and Gravity*

Right now our acceleration is a number we just *pick*. But in the real world, acceleration doesn't
come from nowhere — it comes from **forces**: gravity pulling the drone down, thrust from the
motors pushing it up. In the next notebook you'll learn **Newton's second law** (`a = F / m`) —
intuitively, with animations — and you'll watch a drone **fall under gravity**, then add **thrust**
and try to make it **hover by hand**. You'll discover that hand-balancing is frustratingly fragile…
which is exactly the problem a *controller* will solve. 🚁

**Great work finishing Notebook 01!**
